In [29]:
import numpy as np

### SHAPETRACKER ###

class ShapeTracker:
    def __init__(self, shape) -> None:
        self.shape = shape 
        self.permutes = None
        self.stride = [1]
        self.pad = [0]

### LAZYBUFFER ###

class LazyBuffer:
    def __init__(self) -> None:
        self.buffer = []
        self.grad = 0
    
    def numpy(self):
        return self.forward()
    
    def debug(self):
        print(self.buffer)
    
    def __add__(self, other): return Add(self, other)
    def __mul__(self, other): return Mul(self, other)
    def __matmul__(self, other): return MatMul(self, other)

    def sum(self, axis=None): return Sum(self, axis)


### TENSOR ###

class Tensor(LazyBuffer):  # Tensor is just a realized lazybuffer 
    def __init__(self, value) -> None:
        self.data = np.array(value)
        self.shape = self.data.shape
        self.shapeTrack = ShapeTracker(self.shape)
        self.requires_grad = False
        self.grad = 0

    def forward(self):
        return self.data
    def backward(self, grad):  # Leaf tensors don't need to do anything
        self.grad += grad

    def numpy(self):
        return self.data


### OPERATION TYPES ###

class Unary(LazyBuffer):
    pass


class Binary(LazyBuffer):
    def __init__(self, a, b) -> None:
        super().__init__()
        self.a = Tensor(a) if not isinstance(a, LazyBuffer) else a
        self.b = Tensor(b) if not isinstance(b, LazyBuffer) else b

class Reduce(LazyBuffer):
    def __init__(self, a, axis) -> None:
        super().__init__()
        self.grad = 0
        self.a = Tensor(a) if not isinstance(a, LazyBuffer) else a
        self.axis = axis
        # self.shape = 1 if axis is None else (elm for i, elm in enumerate(self.a.shape) if i != axis)

### OPERATIONS ###

class Add(Binary):
    def forward(self):
        self.data = self.a.forward() + self.b.forward()
        self.shape = self.data.shape
        return self.data

    def backward(self, grad=1):
        self.grad += grad
        self.a.backward(grad)
        self.b.backward(grad)

class Mul(Binary):
    def forward(self):
        self.data = self.a.forward() * self.b.forward()
        self.shape = self.data.shape
        return self.data

    def backward(self, grad=1):
        self.grad += grad
        self.a.backward(grad * self.b.data)
        self.b.backward(grad * self.a.data)

class MatMul(Binary):
    def forward(self):
        self.data = self.a.forward() @ self.b.forward()
        self.shape = self.data.shape
        return self.data

    def backward(self, grad=1):
        self.grad += grad
        self.a.backward(grad @ self.b.data.T)
        self.b.backward(self.a.data.T @ grad)


class Sum(Reduce):
    def forward(self):
        self.data = np.sum(self.a.forward(), axis=self.axis)
        self.shape = self.data.shape
        return self.data

    def backward(self, grad=1):
        self.grad += grad
        self.a.backward(grad * np.ones(self.a.shape))
    

class Rand(Tensor):
    def __init__(self, shape) -> None:
        self.data = np.random.rand(*shape)
        super().__init__(self.data)


### MLOPS ###

class Convolution(LazyBuffer):
    def forward(x, weight, bias, stride, padding):
        pass

### MODULES ###

class Module():
    pass

class Linear(Module):
    def __init__(self, in_features, out_features) -> None:
        self.in_features = in_features
        self.out_features = out_features
        self.weight = Rand((in_features, out_features))
        self.bias = Rand((1, out_features))

    def forward(self, x):
        self.x = x
        return x @ self.weight + self.bias

    def backward(self, grad):
        self.weight.grad += self.x.T @ grad
        self.bias.grad += grad.sum(axis=0)
        return grad @ self.weight.T
    
class Conv2D(Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0) -> None:
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        self.weight = Rand((out_channels, in_channels, kernel_size, kernel_size))
        self.bias = Rand((1, out_channels))

    def forward(self, x):  # Change this, define convolution as reshaped matmul
        assert len(x.shape) == 4, "Input tensor must be 4D (batch, channel, height, width)"
        # result = np.zeros((x.shape[0], self.out_channels, *x.shape[2:]))  
        
        # skip = int(self.kernel_size/2)
        # for i in range(skip, x.shape[2]-skip, self.stride):
        #     for j in range(skip, x.shape[3]-skip, self.stride):
        #         result[:, :, i, j] = x.data[:, :, i-skip:i+skip, j-skip:j+skip] * self.weight

        # return Tensor(result + self.bias)
        return Convolution.forward(x, self.weight, self.bias, self.stride, self.padding)

    def backward(self, grad):
        pass  # Can't be bothered 
        


In [31]:
Linear(3, 2).forward(Rand((2, 3))).numpy()

array([[1.44542334, 1.68473979],
       [1.11385073, 1.18355981]])

In [35]:
conv = Conv2D(3, 2, 3)
conv.forward(Rand((2, 3, 4, 4)))

In [5]:
a = Tensor(np.array([[1, 2], [3, 4]]))
b = Tensor(np.array([[5, 6], [7, 8]]))

c = a + b
d = c * b
e = d + 3
f = e.sum()

print(c)
print(d)
print(e)
print(f)

print(f.numpy())
f.backward()

print(a.grad)
print(b.grad)
print(c.grad)
print(d.grad)
print(e.grad)
print(f.grad)


256
[[5. 6.]
 [7. 8.]]
[[11. 14.]
 [17. 20.]]
[[5. 6.]
 [7. 8.]]
[[1. 1.]
 [1. 1.]]
[[1. 1.]
 [1. 1.]]
1


In [6]:
from torch import *

In [7]:
a = Tensor(np.array([[1, 2], [3, 4]])).requires_grad_()
b = Tensor(np.array([[5, 6], [7, 8]])).requires_grad_()

c = a + b
d = c * b
e = d + 3
f = e.sum()

print(c)
print(d)
print(e)
print(f)

f.backward()

print("-------------------")
print(a.grad)
print(b.grad)
# print(c.grad)
# print(d.grad)
# print(e.grad)
# print(f.grad)


tensor([[ 6.,  8.],
        [10., 12.]], grad_fn=<AddBackward0>)
tensor([[30., 48.],
        [70., 96.]], grad_fn=<MulBackward0>)
tensor([[33., 51.],
        [73., 99.]], grad_fn=<AddBackward0>)
tensor(256., grad_fn=<SumBackward0>)
-------------------
tensor([[5., 6.],
        [7., 8.]])
tensor([[11., 14.],
        [17., 20.]])
